# Урок 18: Захист AI-агентів за допомогою криптографічних квитанцій

## Практична зошит

Цей зошит проходить через чотири вправи:

1. **Підпишіть свою першу квитанцію** для виклику інструменту агента та перевірте її.
2. **Пошкодьте квитанцію** і подивіться, як перевірка не проходить.
3. **Побудуйте ланцюжок з трьох квитанцій** і підтвердьте цілісність ланцюга.
4. **Обгорніть виклик інструменту Microsoft Agent Framework**, щоб кожна дія створювала квитанцію.

Усі криптографічні примітиви імпортуються з добре підтримуваних бібліотек (`pynacl` для Ed25519, `jcs` для RFC 8785 canonical JSON, `hashlib` із стандартної бібліотеки Python для SHA-256). Логіка квитанції — це простий Python, який ви можете читати та змінювати.

Запускайте клітинки по черзі. Кожен розділ короткий і самодостатній.


## Setup

Встановіть дві залежності. Обидві мають дозволяючі ліцензії (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## Допоміжні утиліти

Ці два помічники виконують кодування base64url (без доповнення) і хешування SHA-256 довільних об’єктів. Вони дозволяють зосередити решту блокнота на самій логіці квитанції.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Розділ 1: Підпишіть свій перший чек

Уявіть, що наш агент для **Contoso Travel** щойно знайшов рейси з Сіднея до Лос-Анджелеса для клієнта. Ми хочемо зафіксувати цей виклик інструменту як підписаний чек, щоб майбутній аудитор міг перевірити його без довіри до нас.

### Крок 1.1: Згенеруйте ключ для підпису

У виробництві ключ підпису агента зберігався б у модулі апаратної безпеки (HSM), Azure Key Vault або подібному захищеному сховищі. Для цього уроку ми генеруємо новий ключ у пам’яті.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### Крок 1.2: Створення корисного навантаження квитанції

Корисне навантаження містить усе, що ми хочемо, щоб квитанція засвідчила: хто діяв, який інструмент, з якими аргументами, що повернулося, за якою політикою та коли. Ми хешуємо аргументи та результат замість того, щоб включати їх безпосередньо, щоб квитанція не розкривала конфіденційний вміст.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### Крок 1.3: Підписати та зібрати чек

Три кроки:

1. Канонізувати корисне навантаження, використовуючи JCS, щоб дві реалізації, які створюють однаковий логічний чек, виробляли біт-відповідні байти.
2. Хешувати канонічні байти за допомогою SHA-256.
3. Підписати хеш приватним ключем Ed25519.

Підпис тоді додається до оригінального корисного навантаження для створення остаточного чека.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### Крок 1.4: Перевірте квитанцію

Перевірка робить зворотній процес. Ми видаляємо підпис, повторно обчислюємо канонічний хеш і перевіряємо підпис за допомогою відкритого ключа в квитанції.

Аудитор, який проводить цю перевірку, не потребує від нас нічого, окрім самої квитанції. Ніяких сервісів для виклику, ніяких каталогів ключів для запиту, жодної довіри не потрібно.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

Ви повинні побачити `Receipt is valid: True`. Агент створив свій перший криптографічно підписаний запис аудиту.


## Розділ 2: Змініть чек

Вся суть чеків у тому, що вони є захищеними від підробок. Давайте це доведемо.

Ми змінимо ровно один символ у чеку і подивимось, як перевірка не вдасться.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### Що щойно сталося?

Коли ми змінили `policy_id`, канонічні байти змінилися. SHA-256 хеш цих байтів змінився. Підпис (який був накладений на оригінальний хеш) більше не співпадає з новим хешем. Перевірка коректно повертає `False`.

Немає способу змінити будь-яке поле квитанції і при цьому успішно підтвердити, якщо у зловмисника немає приватного ключа. Доки приватний ключ зберігається у сховищі ключів, а публічний ключ опублікований, фальсифікувати неможливо.

Спробуйте самі: змініть `tool_name` або `agent_id` або `timestamp` у клітинці вище і запустіть знову. Кожна зміна призводить до недійсної квитанції.


## Section 3: Об’єднання квитанцій у ланцюжок

Одна квитанція захищає одну дію. Більшість агентів виконує багато дій. Щоб зробити всю послідовність очевидною для фальсифікації, ми пов’язуємо кожну квитанцію з попередньою, включаючи хеш попередньої квитанції у завантажувальні дані нової квитанції.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Якщо хтось видалить або змінить порядок квитанції, ланцюжок розірветься саме в цій точці. Перевірка будь-якої наступної квитанції не вдасться, оскільки `previous_receipt_hash` більше не співпадає з фактичним хешем її попередника.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Тепер порвіть ланцюг, втручаючись у середній квитанцію, і повторно перевірте. Пошкоджена квитанція не проходить перевірку підпису, І наступна квитанція не проходить перевірку ланцюгового зв’язку (тому що її `previous_receipt_hash` більше не збігається з хешем зміненої середньої квитанції).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Квитанція 0 досі проходить верифікацію (вона не була змінена і не має попередника, від якого залежати). Квитанція 1 не проходить перевірку підпису, оскільки ми змінили `tool_args_hash`. Квитанція 2 не проходить перевірку ланцюжка, оскільки її `previous_receipt_hash` було обчислено на основі оригінальної (тепер зміненої) квитанції 1.

Навіть якщо зловмисник підпише змінену квитанцію 1 повторно (чого він не може зробити без приватного ключа), невідповідність ланцюжка у квитанції 2 все одно виявить підробку. Щоб приховати зміну, зловмиснику довелося б повторно підписати кожну квитанцію від точки зміни і далі, що вимагає володіння приватним ключем.


## Розділ 4: Огорніть виклик інструменту агента підписанням квитанції

У реальному розгортанні ви не хочете, щоб кожен автор агента пам’ятав викликати `make_receipt`. Ви хочете, щоб підписання квитанції було автоматичним для кожного виклику інструменту.

Ось найпростіший шаблон: клас-обгортка, який приймає будь-яку викличну функцію інструменту і повертає варіант, що генерує квитанцію. Це підходить для будь-якого агентського фреймворка, включно з Microsoft Agent Framework (`agent_framework.azure`).

Якщо у вас немає налаштованого проєкту Azure AI Foundry, наведена нижче локальна імітація все одно демонструє цей шаблон.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to AzureAIProjectAgentProvider as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")

### Інтеграція з Microsoft Agent Framework

Обгортка `ReceiptedTool` вище є незалежною від фреймворку. Щоб використовувати її всередині агента, створеного з Microsoft Agent Framework, зареєструйте обгорнуту функцію як інструмент. Приклад (ви заміните заглушку на реєстрацію інструменту Azure AI Foundry):

```python
# Псевдокод, що показує форму інтеграції.
# from agent_framework.azure import AzureAIProjectAgentProvider
# from azure.identity import AzureCliCredential
#
# provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())
# agent = provider.create_agent(
#     instructions="Ви агент Contoso Travel ...",
#     tools=[receipted_lookup],   # обгорнутий інструмент, а не сирий функціонал
# )
# response = agent.run("Знайдіть рейси з Сіднея до Лос-Анджелеса в червні.")
#
# # Після запуску кожен виклик інструменту, зроблений агентом, має підписаний чек:
# audit_chain = receipted_lookup.receipts
```

Фреймворк агента не потребує знань про квитанції. Підписання квитанції обгорнуте навколо інструмента, а не вбудоване у фреймворк. Саме так ви додаєте походження до існуючого коду агента без переписування агента.


## Підсумки та додаткове завдання

Ви:

- Згенерували пару ключів Ed25519.
- Створили та підписали квитанцію для виклику агентського інструменту.
- Перевірили квитанцію офлайн, використовуючи лише відкритий ключ.
- Змінили квитанцію та спостерігали несправність перевірки.
- Створили послідовність із трьох квитанцій, пов’язаних хеш-ланцюгом.
- Змінили середину ланцюга та спостерігали як помилку підпису, так і помилку зв’язку ланцюга.
- Обгорнули функцію інструменту автоматичним підписанням квитанції.

**Додаткове завдання.** Розширте схему квитанції полем `request_id` (UUID для розподіленого трасування). Оновіть `make_receipt`, щоб включити його, і переконайтеся, що квитанції все ще верифікуються повністю. Потім змініть поле після підписання та переконайтеся, що перевірка не пройшла. Це змусить вас усвідомити, як кожен байт канонічного кодування впливає на підпис.

**Важлива межа.** Квитанції доводять три речі і тільки три: атрибуцію (цей ключ підписав цей вміст), цілісність (вміст не змінився після підписання) і порядок (ця квитанція йде після тієї квитанції). Вони НЕ доводять, що дія агента була правильною, що політика, названа в `policy_id`, дійсно була оцінена, або що агент дотримувався всіх правил. Квитанції — це основа. Управління — це система, яку ви будуєте зверху.

Ще раз прочитайте README уроку з урахуванням цієї межі. Найпоширеніша помилка команд із квитанціями — припускати, що «ми маємо квитанції» означає «над нами є управління». Це не так. Квитанції роблять поведінку агента аудиторською. Вони не роблять її правильною.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Відмова від відповідальності**:
Цей документ було перекладено за допомогою сервісу штучного інтелекту для перекладу [Co-op Translator](https://github.com/Azure/co-op-translator). Хоча ми прагнемо до точності, будь ласка, майте на увазі, що автоматичні переклади можуть містити помилки або неточності. Оригінальний документ рідною мовою слід вважати авторитетним джерелом. Для критично важливої інформації рекомендується професійний людський переклад. Ми не несемо відповідальності за будь-які непорозуміння або неправильні тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
